# IndyCar metrics generator

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import numpy as np
import pandas as pd

from types import TracebackType
from pathlib import Path

from numpy.polynomial.polynomial import polyfit

In [ ]:
BASE_DIR = Path("/content/drive/MyDrive/indycar/datasets")
EXTRA_DIR = Path("/content/drive/MyDrive/indycar/scraped_data")

In [ ]:
def computeELO(elos, ranks, k):
  n = len(elos)
  if n <=1:
    return elos

  scores = (n - ranks) / (n-1)

  diff = (elos.reshape(-1,1) - elos.reshape(1,-1)) /400
  expected = 1/(1+10**(-diff))
  np.fill_diagonal(expected, np.nan)
  expected = np.nanmean(expected, axis=1)

  return elos + k * (scores - expected)

In [ ]:
df = pd.read_csv(BASE_DIR / "IndyCar_dataset_v24.csv")
pit_stops_df = pd.read_csv(EXTRA_DIR / "pit_stops.csv")
race_metadata_df = pd.read_csv(EXTRA_DIR / "race_metadata.csv")
cautions_df = pd.read_csv(EXTRA_DIR / "cautions.csv")
lap_positions_df = pd.read_csv(EXTRA_DIR / "lap_positions.csv")
lap_times_df = pd.read_csv(EXTRA_DIR / "lap_times.csv")

In [ ]:
event_id_map = {
    "005_streets_of_st_petersburg_2013": 2645,
    "006_barber_motorsports_park_2013": 2654,
    "007_streets_of_long_beach_2013": 2682,
    "008_streets_of_sao_paulo_2013": 2695,
    "009_indianapolis_motor_speedway_2013": 2704,
    "012_texas_motor_speedway_2013": 2739,
    "013_milwaukee_mile_2013": 2743,
    "014_iowa_speedway_2013": 2753,
    "015_pocono_raceway_2013": 2762,
    "016_streets_of_toronto_2013": 2788,
    "017_streets_of_toronto_2013": 2769,
    "018_mid-ohio_sports_car_course_2013": 2799,
    "019_sonoma_raceway_2013": 2827,
    "020_streets_of_baltimore_2013": 2835,
    "021_reliant_park_2013": 2864,
    "022_reliant_park_2013": 2869,
    "023_auto_club_speedway_2013": 2877,
    "024_streets_of_st_petersburg_2014": 2888,
    "025_streets_of_long_beach_2014": 2911,
    "026_barber_motorsports_park_2014": 2924,
    "027_indianapolis_motor_speedway_2014": 2943,
    "028_indianapolis_motor_speedway_2014": 2988,
    "029_the_raceway_at_belle_isle_park_2014": 2993,
    "030_the_raceway_at_belle_isle_park_2014": 2998,
    "031_texas_motor_speedway_2014": 3002,
    "032_reliant_park_2014": 3009,
    "033_reliant_park_2014": 3014,
    "034_pocono_raceway_2014": 3024,
    "035_iowa_speedway_2014": 3031,
    "036_streets_of_toronto_2014": 3041,
    "037_streets_of_toronto_2014": 3036,
    "038_mid-ohio_sports_car_course_2014": 3056,
    "039_milwaukee_mile_2014": 3082,
    "040_sonoma_raceway_2014": 3106,
    "041_auto_club_speedway_2014": 3113,
    "042_streets_of_st_petersburg_2015": 3149,
    "043_nola_motorsports_park_2015": 3173,
    "044_streets_of_long_beach_2015": 3192,
    "045_barber_motorsports_park_2015": 3205,
    "046_indianapolis_motor_speedway_2015": 3226,
    "047_indianapolis_motor_speedway_2015": 3236,
    "048_the_raceway_at_belle_isle_park_2015": 3276,
    "049_the_raceway_at_belle_isle_park_2015": 3282,
    "050_texas_motor_speedway_2015": 3285,
    "051_streets_of_toronto_2015": 3292,
    "052_auto_club_speedway_2015": 3315,
    "053_milwaukee_mile_2015": 3323,
    "054_iowa_speedway_2015": 3327,
    "055_mid-ohio_sports_car_course_2015": 3341,
    "056_pocono_raceway_2015": 3361,
    "057_sonoma_raceway_2015": 3367,
    "058_streets_of_st_petersburg_2016": 3409,
    "059_phoenix_raceway_2016": 3413,
    "060_streets_of_long_beach_2016": 3423,
    "061_barber_motorsports_park_2016": 3447,
    "062_indianapolis_motor_speedway_2016": 3471,
    "063_indianapolis_motor_speedway_2016": 3484,
    "064_the_raceway_at_belle_isle_park_2016": 3512,
    "065_the_raceway_at_belle_isle_park_2016": 3518,
    "066_road_america_2016": 3527,
    "067_iowa_speedway_2016": 3557,
    "068_streets_of_toronto_2016": 3563,
    "069_mid-ohio_sports_car_course_2016": 3600,
    "070_pocono_raceway_2016": 3608,
    "071_texas_motor_speedway_2016": 3521,
    "072_watkins_glen_international_2016": 3616,
    "073_sonoma_raceway_2016": 3645,
    "074_streets_of_st_petersburg_2017": 3651,
    "075_streets_of_long_beach_2017": 3678,
    "076_barber_motorsports_park_2017": 3686,
    "077_phoenix_raceway_2017": 3703,
    "078_indianapolis_motor_speedway_2017": 3707,
    "079_indianapolis_motor_speedway_2017": 3740,
    "080_the_raceway_at_belle_isle_park_2017": 3748,
    "081_the_raceway_at_belle_isle_park_2017": 3752,
    "082_texas_motor_speedway_2017": 3755,
    "083_road_america_2017": 3762,
    "084_iowa_speedway_2017": 3783,
    "085_streets_of_toronto_2017": 3796,
    "086_mid-ohio_sports_car_course_2017": 3815,
    "087_pocono_raceway_2017": 3838,
    "088_world_wide_technology_raceway_2017": 3842,
    "089_watkins_glen_international_2017": 3848,
    "090_sonoma_raceway_2017": 3857,
    "091_streets_of_st_petersburg_2018": 5230,
    "092_phoenix_raceway_2018": 5229,
    "093_streets_of_long_beach_2018": 5232,
    "094_barber_motorsports_park_2018": 5231,
    "095_indianapolis_motor_speedway_2018": 5226,
    "096_indianapolis_motor_speedway_2018": 5225,
    "097_the_raceway_at_belle_isle_park_2018": 5228,
    "098_the_raceway_at_belle_isle_park_2018": 5227,
    "099_texas_motor_speedway_2018": 5233,
    "100_road_america_2018": 5239,
    "101_iowa_speedway_2018": 5238,
    "102_streets_of_toronto_2018": 5241,
    "103_mid-ohio_sports_car_course_2018": 5240,
    "104_pocono_raceway_2018": 5235,
    "105_world_wide_technology_raceway_2018": 5234,
    "106_portland_international_raceway_2018": 5237,
    "107_sonoma_raceway_2018": 5236,
    "108_streets_of_st_petersburg_2019": 5463,
    "109_circuit_of_the_americas_2019": 5477,
    "110_barber_motorsports_park_2019": 5475,
    "111_streets_of_long_beach_2019": 5476,
    "112_indianapolis_motor_speedway_2019": 5474,
    "113_indianapolis_motor_speedway_2019": 5473,
    "114_the_raceway_at_belle_isle_park_2019": 5472,
    "115_the_raceway_at_belle_isle_park_2019": 5478,
    "116_texas_motor_speedway_2019": 5471,
    "117_road_america_2019": 5469,
    "118_streets_of_toronto_2019": 5467,
    "119_iowa_speedway_2019": 5468,
    "120_mid-ohio_sports_car_course_2019": 5466,
    "121_pocono_raceway_2019": 5465,
    "122_world_wide_technology_raceway_2019": 5464,
    "123_portland_international_raceway_2019": 5470,
    "124_weathertech_raceway_laguna_seca_2019": 5479,
    "125_texas_motor_speedway_2020": 5733,
    "126_indianapolis_motor_speedway_2020": 5741,
    "127_road_america_2020": 5742,
    "128_road_america_2020": 5748,
    "129_iowa_speedway_2020": 5751,
    "130_iowa_speedway_2020": 5754,
    "131_indianapolis_motor_speedway_2020": 5777,
    "132_world_wide_technology_raceway_2020": 5780,
    "133_world_wide_technology_raceway_2020": 5782,
    "134_mid-ohio_sports_car_course_2020": 5762,
    "135_mid-ohio_sports_car_course_2020": 5765,
    "136_indianapolis_motor_speedway_2020": 5783,
    "137_indianapolis_motor_speedway_2020": 5784,
    "138_streets_of_st_petersburg_2020": 5790,
    "139_barber_motorsports_park_2021": 5797,
    "140_streets_of_st_petersburg_2021": 5798,
    "141_texas_motor_speedway_2021": 5810,
    "142_texas_motor_speedway_2021": 5809,
    "143_indianapolis_motor_speedway_2021": 5808,
    "144_indianapolis_motor_speedway_2021": 5807,
    "145_the_raceway_at_belle_isle_park_2021": 5806,
    "146_the_raceway_at_belle_isle_park_2021": 5812,
    "147_road_america_2021": 5813,
    "148_mid-ohio_sports_car_course_2021": 5801,
    "149_streets_of_nashville_2021": 5800,
    "150_indianapolis_motor_speedway_2021": 5802,
    "151_world_wide_technology_raceway_2021": 5799,
    "152_portland_international_raceway_2021": 5805,
    "153_weathertech_raceway_laguna_seca_2021": 5803,
    "154_streets_of_long_beach_2021": 5811,
    "155_streets_of_st_petersburg_2022": 5921,
    "156_texas_motor_speedway_2022": 5912,
    "157_streets_of_long_beach_2022": 5906,
    "158_barber_motorsports_park_2022": 5907,
    "160_indianapolis_motor_speedway_2022": 5909,
    "161_the_raceway_at_belle_isle_park_2022": 5910,
    "162_road_america_2022": 5911,
    "163_mid-ohio_sports_car_course_2022": 5913,
    "164_streets_of_toronto_2022": 5920,
    "165_iowa_speedway_2022": 5914,
    "166_iowa_speedway_2022": 5915,
    "167_indianapolis_motor_speedway_2022": 5916,
    "168_streets_of_nashville_2022": 5917,
    "169_world_wide_technology_raceway_2022": 5918,
    "170_portland_international_raceway_2022": 5919,
    "171_weathertech_raceway_laguna_seca_2022": 5905,
    "172_streets_of_st_petersburg_2023": 6125,
    "173_texas_motor_speedway_2023": 6139,
    "174_streets_of_long_beach_2023": 6138,
    "175_barber_motorsports_park_2023": 6137,
    "176_indianapolis_motor_speedway_2023": 6136,
    "177_indianapolis_motor_speedway_2023": 6135,
    "178_streets_of_detroit_2023": 6134,
    "179_road_america_2023": 6140,
    "180_mid-ohio_sports_car_course_2023": 6133,
    "181_streets_of_toronto_2023": 6131,
    "182_iowa_speedway_2023": 6130,
    "183_iowa_speedway_2023": 6129,
    "184_streets_of_nashville_2023": 6127,
    "185_indianapolis_motor_speedway_2023": 6128,
    "186_world_wide_technology_raceway_2023": 6126,
    "187_portland_international_raceway_2023": 6132,
    "188_weathertech_raceway_laguna_seca_2023": 6141,
    "189_streets_of_st_petersburg_2024": 6298,
    "190_streets_of_long_beach_2024": 6297,
    "191_barber_motorsports_park_2024": 6312,
    "192_indianapolis_motor_speedway_2024": 6301,
    "193_indianapolis_motor_speedway_2024": 6309,
    "194_streets_of_detroit_2024": 6296,
    "195_road_america_2024": 6308,
    "196_weathertech_raceway_laguna_seca_2024": 6311,
    "197_mid-ohio_sports_car_course_2024": 6303,
    "198_iowa_speedway_2024": 6310,
    "199_iowa_speedway_2024": 6307,
    "200_streets_of_toronto_2024": 6306,
    "201_world_wide_technology_raceway_2024": 6302,
    "202_portland_international_raceway_2024": 6305,
    "203_milwaukee_mile_2024": 6299,
    "204_milwaukee_mile_2024": 6313,
    "205_nashville_superspeedway_2024": 6304,
    "206_streets_of_st_petersburg_2025": 4444,
    "207_the_thermal_club_2025": 6463,
    "208_streets_of_long_beach_2025": 6457,
    "209_barber_motorsports_park_2025": 6458,
    "210_indianapolis_motor_speedway_2025": 6459,
    "211_indianapolis_motor_speedway_2025": 6460,
    "212_streets_of_detroit_2025": 6461,
    "213_world_wide_technology_raceway_2025": 6462,
    "214_mid-ohio_sports_car_course_2025": 6464,
    "215_iowa_speedway_2025": 6465,
    "216_iowa_speedway_2025": 6466,
    "217_streets_of_toronto_2025": 6467,
    "218_weathertech_raceway_laguna_seca_2025": 6468,
    "219_portland_international_raceway_2025": 6469,
    "220_milwaukee_mile_2025": 6470,
    "221_nashville_superspeedway_2025": 6471,
    "222_streets_of_st_petersburg_2026": 6472,
    "223_phoenix_raceway_2026": 6473,
    "224_streets_of_arlington_2026": 6474,
    "225_barber_motorsports_park_2026": 6475,
    "226_streets_of_long_beach_2026": 6476,
    "227_indianapolis_motor_speedway_2026": 6744,
    "228_indianapolis_motor_speedway_2026": 6743,
    "229_streets_of_detroit_2026": 6742,
    "230_world_wide_technology_raceway_2026": 6741,
    "231_road_america_2026": 6747,
    "232_mid-ohio_sports_car_course_2026": 6740,
    "233_nashville_superspeedway_2026": 6748,
    "234_portland_international_raceway_2026": 6733,
    "235_streets_of_markham_2026": 6737,
    "236_streets_of_washington_2026": 6796,
    "237_milwaukee_mile_2026": 6739,
    "238_milwaukee_mile_2026": 6738,
}

In [ ]:
pit_stops_df["event_id"] = pit_stops_df["event_id"].replace(event_id_map)
race_metadata_df["event_id"] = race_metadata_df["event_id"].replace(event_id_map)
cautions_df["event_id"] = cautions_df["event_id"].replace(event_id_map)
lap_positions_df["event_id"] = lap_positions_df["event_id"].replace(event_id_map)
lap_times_df["event_id"] = lap_times_df["event_id"].replace(event_id_map)

valid_events = set(race_metadata_df["event_id"].unique())
pit_stops_df = pit_stops_df[pit_stops_df["event_id"].isin(valid_events)]
lap_positions_df = lap_positions_df[lap_positions_df["event_id"].isin(valid_events)]
lap_times_df = lap_times_df[lap_times_df["event_id"].isin(valid_events)]
cautions_df = cautions_df[cautions_df["event_id"].isin(valid_events)]

In [ ]:
pit_stops_df["DriverName"] = pit_stops_df["FName"] + " " + pit_stops_df["LName"]
pit_stops_df = pit_stops_df.drop(columns=["FName", "LName"])

lap_positions_df["DriverName"] = lap_positions_df["FName"] + " " + lap_positions_df["LName"]
lap_positions_df = lap_positions_df.drop(columns=["FName", "LName"])

lap_times_df["DriverName"] = lap_times_df["FName"] + " " + lap_times_df["LName"]
lap_times_df = lap_times_df.drop(columns=["FName", "LName"])

name_fixes = {
    "Patricio O'Ward": "Pato O'Ward",
    "Rinus Veekay": "Rinus VeeKay",
    "EJ Viso": "E.J. Viso",
    "Simona de Silvestro": "Simona De Silvestro",
}

pit_stops_df["DriverName"] = pit_stops_df["DriverName"].replace(name_fixes)
lap_positions_df["DriverName"] = lap_positions_df["DriverName"].replace(name_fixes)
lap_times_df["DriverName"] = lap_times_df["DriverName"].replace(name_fixes)

In [ ]:
pit_meta_df = pit_stops_df.merge(
    race_metadata_df[["event_id", "TotalRaceLaps"]], on="event_id", how="left"
)

In [ ]:
ritmo = 5
DNFwindow = 20
baseELO = 1500.0
k = 64
tk = 48
ek = 32

ordered_cols = [
    "DriverName", "DriverID", "Rookie",
    "DRFAvg", "DTAvg", "DTTAvg", "DNFRate", "TDNFRate",
    "DriverElo", "DriverTElo", "DriverTTElo", "DriverRitmo",
    "PositionStart", "StartVsField",
    "TeamName", "TeamID", "TRP", "TTP", "TeamDNFRate",
    "TeamElo", "TeamTElo", "TeamRitmo",
    "TeamTrackAvgPitStops", "TeamTrackAvgPittedCautionPCT",
    "CarEngine", "EngineID", "EngineElo", "EngineTElo", "EngineTTElo",
    "EventName", "Track", "TrackID",
    "EventTrackType", "EventTrackTypeID", "TrackAvgCautions", "TrackAvgCautionLaps",
    "TrackTypeAvgCautions", "TrackTypeAvgCautionLaps",  "TrackAvgSpeed",
    "EventDate", "EventDateFormatted", "EventID",
    "Era", "EraID",
    "Status", "StatusID", "FieldSize",
    "TotalCautions", "TotalCautionLaps", "TotalLeadChanges",
    "TotalRaceLaps", "TrackLength", "AvgRaceSpeed", "FastestLapSpeed",
    "AvgStintLength", "StintVariance",
    "FirstStopLap", "FirstStopVSAvgFirstStop",
    "PittedCautions", "PitStops", "PittedCautionPCT",
    "TotalPitStops", "PitStrategy",
    "BestRacePosition", "WorstRacePosition",
    "LapsLed", "Top5Laps", "Top10Laps", "PositionVolatility",
    "position_lap1", "PositionsGainedLap1",
    "PositionsGainedPits", "PositionsGainedCaution",
    "BestLapSpeed", "LapConsistency",
    "PaceDeg", "AvgPaceVSLeadPace",
    "AvgPitTime", "PitAvgTimeVSOverall",
    "TrackAvgPitStops", "FuelWindowEstimate",
    "PositionFinish", "NormalizedPositionFinish",
]

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

df["NormalizedPositionFinish"] = (
    (df["PositionFinish"]-1)/(df["FieldSize"]-1)
)

In [ ]:
df["DRFAvg"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

df["DTAvg"] = (
    df.groupby(["DriverID", "TrackID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

df['DTTAvg'] = (
    df.groupby(["DriverID", "EventTrackTypeID"])["NormalizedPositionFinish"].transform(lambda s: s.shift(1).expanding().mean())
)

df["DNFRate"] = (
    df.groupby("DriverID")["StatusID"].transform((lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean()))
)

df["TDNFRate"] = (
    df.groupby(["DriverID", "TrackID"])["StatusID"].transform((lambda s: s.shift(1).expanding().mean()))
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID"]).reset_index(drop=True)

driverElo ={}
df["DriverElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
  indx = event.index
  drivers = event["DriverID"].values
  ranks = event["PositionFinish"].values.astype(float)

  preElo = np.array([driverElo.get(d, baseELO) for d in drivers], dtype=float)
  df.loc[indx, "DriverElo"] = preElo

  postElo = computeELO(preElo, ranks, k)
  for d, new in zip(drivers, postElo):
    driverElo[d] = float(new)


driverTrackElo = {}
df["DriverTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracks = event["TrackID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackElo.get((d, t), baseELO) for d, t in zip(drivers, tracks)], dtype=float)
    df.loc[indx, "DriverTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, t, new in zip(drivers, tracks, postElo):
        driverTrackElo[(d, t)] = float(new)

driverTrackTypeElo = {}
df["DriverTTElo"] = np.nan

for eventID, event in df.groupby("EventID", sort=False):
    indx = event.index
    drivers = event["DriverID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["PositionFinish"].values.astype(float)

    preElo = np.array([driverTrackTypeElo.get((d, tt), baseELO) for d, tt in zip(drivers, tracktypes)], dtype=float)
    df.loc[indx, "DriverTTElo"] = preElo

    postElo = computeELO(preElo, ranks, k)
    for d, tt, new in zip(drivers, tracktypes, postElo):
        driverTrackTypeElo[(d, tt)] = float(new)

In [ ]:
df["DriverRitmo"] = (
    df.groupby("DriverID")["NormalizedPositionFinish"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
df = df.drop(columns=["TRP", "TTP", "TeamDNFRate", "TeamElo",  "TeamTElo", "TeamRitmo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

teamdf = (
    df.groupby(["EventID", "TeamID"], as_index=False)
    .agg(
        teamPerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
        teamDNF=("StatusID", "max")
    ).sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)
)

teamdf = teamdf.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

In [ ]:
teamdf["TRP"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).rolling(ritmo, min_periods=1).mean())
)

teamdf["TTP"] = (
    teamdf.groupby(["TeamID", "EventTrackTypeID"])["teamPerf"].transform(lambda s: s.shift(1).expanding().mean())
)

teamdf["TeamDNFRate"] = (
     teamdf.groupby("TeamID")["teamDNF"].transform(lambda s: s.shift(1).rolling(DNFwindow, min_periods=1).mean())
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID"]).reset_index(drop=True)

teamElo ={}
teamdf["TeamElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
  indx = event.index
  teams = event["TeamID"].values
  ranks = event["teamPerf"].rank(method="first").values.astype(float)

  preElo = np.array([teamElo.get(t, baseELO) for t in teams], dtype=float)
  teamdf.loc[indx, "TeamElo"] = preElo

  postElo = computeELO(preElo, ranks, tk)
  for t, new in zip(teams, postElo):
    teamElo[t] = float(new)

teamTrackElo = {}
teamdf["TeamTElo"] = np.nan

for eventID, event in teamdf.groupby("EventID", sort=False):
    indx = event.index
    teams = event["TeamID"].values
    tracks = event["TrackID"].values
    ranks = event["teamPerf"].rank(method="first").values.astype(float)

    preElo = np.array([teamTrackElo.get((t, tr), baseELO) for t, tr in zip(teams, tracks)], dtype=float)
    teamdf.loc[indx, "TeamTElo"] = preElo

    postElo = computeELO(preElo, ranks, tk)
    for t, tr, new in zip(teams, tracks, postElo):
        teamTrackElo[(t, tr)] = float(new)

teamdf["TeamRitmo"] = (
    teamdf.groupby("TeamID")["teamPerf"].transform(lambda s: s.shift(1).ewm(ritmo, min_periods=1).mean())
)

In [ ]:
df = df.merge(
    teamdf[["EventID", "TeamID", "TRP", "TTP", "TeamDNFRate", "TeamElo", "TeamTElo", "TeamRitmo"]],
    on=["EventID", "TeamID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "TeamID", "DriverID"]).reset_index(drop=True)

In [ ]:
df = df.drop(columns=["EngineElo", "EngineTElo", "EngineTTElo"])
df["EventDate"] = pd.to_datetime(df["EventDate"])

enginedf = (
    df.groupby(["EventID", "EngineID"], as_index=False)
    .agg(
        enginePerf=("NormalizedPositionFinish", "mean"),
        EventTrackTypeID=("EventTrackTypeID", "first"),
        TrackID=("TrackID", "first"),
        EventDate=("EventDate", "first"),
    ).sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)
)

In [ ]:
df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID"]).reset_index(drop=True)

engineElo ={}
enginedf["EngineElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
  indx = event.index
  engines = event["EngineID"].values
  ranks = event["enginePerf"].rank(method="first").values.astype(float)

  preElo = np.array([engineElo.get(e, baseELO) for e in engines], dtype=float)
  enginedf.loc[indx, "EngineElo"] = preElo

  postElo = computeELO(preElo, ranks, ek)
  for e, new in zip(engines, postElo):
    engineElo[e] = float(new)

engineTrackElo = {}
enginedf["EngineTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracks = event["TrackID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackElo.get((e, tr), baseELO) for e, tr in zip(engines, tracks)], dtype=float)
    enginedf.loc[indx, "EngineTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tr, new in zip(engines, tracks, postElo):
        engineTrackElo[(e, tr)] = float(new)

engineTrackTypeElo = {}
enginedf["EngineTTElo"] = np.nan

for eventID, event in enginedf.groupby("EventID", sort=False):
    indx = event.index
    engines = event["EngineID"].values
    tracktypes = event["EventTrackTypeID"].values
    ranks = event["enginePerf"].rank(method="first").values.astype(float)

    preElo = np.array([engineTrackTypeElo.get((e, tt), baseELO) for e, tt in zip(engines, tracktypes)], dtype=float)
    enginedf.loc[indx, "EngineTTElo"] = preElo

    postElo = computeELO(preElo, ranks, ek)
    for e, tt, new in zip(engines, tracktypes, postElo):
        engineTrackTypeElo[(e, tt)] = float(new)

In [ ]:
df = df.merge(
    enginedf[["EventID", "EngineID", "EngineElo", "EngineTElo", "EngineTTElo"]],
    on=["EventID", "EngineID"],
    how="left"
)

df["EventDate"] = pd.to_datetime(df["EventDate"])
df = df.sort_values(["EventDate", "EventID", "EngineID", "DriverID"]).reset_index(drop=True)

In [ ]:
def compute_stint_metrics(group):
  group = group.sort_values("StopNumber")
  stop_laps = group["DriverLap"].values
  total_laps = group["TotalRaceLaps"].iloc[0]

  bounds = np.concatenate([[0], stop_laps, [total_laps]])
  stint_lengths = np.diff(bounds)

  return pd.Series({
      "TotalPitStops": group["TotalPitStops"].iloc[0],
      "TotalRaceLaps": total_laps,
      "AvgStintLength": stint_lengths.mean(),
      "StintVariance": stint_lengths.var(),
  })

stint_metrics_df = (
  pit_meta_df.groupby(["event_id", "DriverName", "carNum"])
  .apply(compute_stint_metrics, include_groups=False)
  .reset_index()
)

In [ ]:
def count_caution_pit(group):
  event = group.name[0]
  stop_laps = group.sort_values("StopNumber")["DriverLap"].values

  event_cautions = cautions_df[cautions_df["event_id"] == event]
  cautions_pit = 0
  for lap in stop_laps:
    is_under_caution = (
        (event_cautions["start_lap"] <= lap) &
        (lap <= event_cautions["end_lap"])
    ).any()
    if is_under_caution:
      cautions_pit += 1

  return pd.Series({
      "PittedCautions": cautions_pit,
      "PitStops": len(stop_laps),
      "PittedCautionPCT": cautions_pit / len(stop_laps) if len(stop_laps) > 0 else 0
  })

caution_pit_df = (
    pit_meta_df
    .groupby(["event_id", "DriverName", "carNum"])
    .apply(count_caution_pit, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_position_summary(group):
    positions = group.sort_values("lap")["position"]

    return pd.Series({
        "BestRacePosition": positions.min(),
        'WorstRacePosition': positions.max(),
        "LapsLed": (positions == 1).sum(),
        "Top5Laps": (positions <= 5).sum(),
        "Top10Laps": (positions <= 10).sum(),
        "PositionVolatility": positions.std(),
    })

position_summary = (
    lap_positions_df
    .groupby(["event_id", "DriverName","car"])
    .apply(compute_position_summary, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pit_cycle_gains(group):
    event = group.name[0]
    car = group.name[2]

    stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ].sort_values("StopNumber")

    if len(stops) == 0:
        return pd.Series({"PositionsGainedPits": 0})

    total_gain = 0
    for _, stop in stops.iterrows():
        stop_lap = stop["DriverLap"]

        before = group[group["lap"] == stop_lap - 1]["position"]
        after = group[group["lap"] == stop_lap + 10]["position"]

        if len(before) > 0 and len(after) > 0:
            total_gain += before.iloc[0] - after.iloc[0]

    return pd.Series({"PositionsGainedPits": total_gain})

pit_cycle_gains = (
    lap_positions_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pit_cycle_gains, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_caution_gains(group):
    event = group.name[0]

    event_cautions = cautions_df[cautions_df["event_id"] == event]

    if len(event_cautions) == 0:
        return pd.Series({"PositionsGainedCaution": 0})

    total_gain = 0
    for _, caution in event_cautions.iterrows():
        start = caution["start_lap"]
        end = caution["end_lap"]

        before = group[group["lap"] == start - 1]["position"]
        after = group[group["lap"] == end + 1]["position"]

        if len(before) > 0 and len(after) > 0:
            total_gain += before.iloc[0] - after.iloc[0]

    return pd.Series({"PositionsGainedCaution": total_gain})

caution_gains = (
    lap_positions_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_caution_gains, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pace_summary(group):
  event = group.name[0]
  car = group.name[2]

  group = group.sort_values("lap")

  pit_stops = pit_meta_df[
      (pit_meta_df["event_id"] == event) &
      (pit_meta_df["carNum"] == car)
  ]["DriverLap"].values

  race_cautions = cautions_df[cautions_df["event_id"] == event]
  caution_laps = set()
  for _, c in race_cautions.iterrows():
    for lap in range(c["start_lap"], c["end_lap"] + 1):
      caution_laps.add(lap)

  green = group[
      (~group["lap"].isin(pit_stops)) &
      (~group["lap"].isin(caution_laps))]["lap_time"]

  return pd.Series({
      "BestLapSpeed": group["lap_time"].min(),
      "LapConsistency": green.std() if len(green) > 0 else np.nan
  })

pace_summary = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_summary, include_groups=False)
  .reset_index()
)

In [ ]:
def compute_pace_degradation(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ].sort_values("StopNumber")["DriverLap"].values

    total_laps = race_metadata_df[race_metadata_df["event_id"] == event]["TotalRaceLaps"].iloc[0]
    boundaries = np.concatenate([[1], pit_stops, [total_laps]])

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    slopes = []
    for i in range(len(boundaries) - 1):
        stint_start = boundaries[i]
        stint_end = boundaries[i + 1]

        stint = group[
            (group["lap"] >= stint_start) &
            (group["lap"] < stint_end) &
            (~group["lap"].isin(pit_stops)) &
            (~group["lap"].isin(caution_laps))
        ]

        if len(stint) >= 3:
            coeffs = polyfit(stint["lap"].values, stint["lap_time"].values, 1)
            slopes.append(coeffs[1])

    return pd.Series({
        "PaceDeg": np.mean(slopes) if len(slopes) > 0 else np.nan
    })

pace_degradation = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_degradation, include_groups=False)
    .reset_index()
)

In [ ]:
# 10 min
def compute_pace_vs_leader(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    race_laps = lap_times_df[lap_times_df["event_id"] == event]
    leader_pace = race_laps.groupby("lap")["lap_time"].min()

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ]["DriverLap"].values


    green = group[
        (~group['lap'].isin(pit_stops)) &
        (~group['lap'].isin(caution_laps))
    ]

    gaps = []
    for _, row in green.iterrows():
        lap_num = row["lap"]
        if lap_num in leader_pace.index and lap_num not in caution_laps:
            gap = row["lap_time"] - leader_pace[lap_num]
            gaps.append(gap)

    return pd.Series({
        "AvgPaceVSLeadPace": np.mean(gaps) if len(gaps) > 0 else np.nan
    })

pace_vs_leader = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pace_vs_leader, include_groups=False)
    .reset_index()
)

In [ ]:
def compute_pit_duration(group):
    event = group.name[0]
    car = group.name[2]

    group = group.sort_values("lap")

    pit_stops = pit_meta_df[
        (pit_meta_df["event_id"] == event) &
        (pit_meta_df["carNum"] == car)
    ]["DriverLap"].values

    race_cautions = cautions_df[cautions_df["event_id"] == event]
    caution_laps = set()
    for _, c in race_cautions.iterrows():
        for lap in range(c["start_lap"], c["end_lap"] + 1):
            caution_laps.add(lap)

    green = group[
        (~group["lap"].isin(pit_stops)) &
        (~group["lap"].isin(caution_laps))
    ]["lap_time"]

    normal_time = green.median()

    pit_lap_times = group[group["lap"].isin(pit_stops)]["lap_time"]
    pit_durations = pit_lap_times - normal_time

    return pd.Series({
        "AvgPitTime": pit_durations.mean() if len(pit_durations) > 0 else np.nan,
        "PitAvgTimeVSOverall": np.nan  # placeholder, computed after
    })

pit_duration = (
    lap_times_df
    .groupby(["event_id", "DriverName", "car"])
    .apply(compute_pit_duration, include_groups=False)
    .reset_index()
)

field_avg_pit = (
    pit_duration
    .groupby("event_id")["AvgPitTime"]
    .transform("mean")
)

pit_duration["PitAvgTimeVSOverall"] = pit_duration["AvgPitTime"] - field_avg_pit

In [ ]:
pit_strategy_map = {1: '1-stop', 2: '2-stop', 3: '3-stop', 4: '4-stop', 5: '5-stop'}

strategy_type_df = (
    pit_meta_df
    .groupby(["event_id", "DriverName", "carNum"])["TotalPitStops"]
    .first()
    .reset_index()
)

strategy_type_df["PitStrategy"] = strategy_type_df["TotalPitStops"].map(pit_strategy_map).fillna("Other")

first_stop_df = (
    pit_meta_df
    .sort_values("StopNumber")
    .groupby(["event_id", "DriverName", "carNum"])
    .first()
    .reset_index()
    [["event_id", "DriverName", "carNum", "DriverLap"]]
    .rename(columns={"DriverLap": "FirstStopLap"})
)

field_avg_first_stop_df = (
    first_stop_df
    .groupby("event_id")["FirstStopLap"]
    .transform("mean")
)

first_stop_df["FirstStopVSAvgFirstStop"] = first_stop_df["FirstStopLap"] - field_avg_first_stop_df

In [ ]:
lap1_positions = lap_positions_df[lap_positions_df['lap'] == 1][['event_id', 'car', 'DriverName', 'position']]
lap1_positions = lap1_positions.rename(columns={'position': 'position_lap1'})

In [ ]:
race_metadata_df.rename(columns={"event_id": "EventID"}, inplace=True)

for frame in [race_metadata_df, stint_metrics_df, first_stop_df, caution_pit_df,
              strategy_type_df, position_summary, lap1_positions, pit_cycle_gains,
              caution_gains, pace_summary, pace_degradation, pace_vs_leader, pit_duration]:
    frame.rename(columns={"event_id": "EventID"}, inplace=True)

In [ ]:
# Race metadata — one row per race, every driver gets the same values
df = df.merge(
    race_metadata_df[["EventID", "TotalCautions", "TotalCautionLaps",
                       "TotalLeadChanges", "TotalRaceLaps", "TrackLength",
                       "AvgRaceSpeed", "FastestLapSpeed"]],
    on="EventID",
    how="left"
)

# Stint metrics (avg_stint_length, stint_length_variance)
df = df.merge(
    stint_metrics_df[["EventID", "DriverName", "AvgStintLength", "StintVariance"]],
    on=["EventID", "DriverName"],
    how="left"
)

# First stop + first stop vs field avg
df = df.merge(
    first_stop_df[["EventID", "DriverName", "FirstStopLap", "FirstStopVSAvgFirstStop"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Caution pit metrics
df = df.merge(
    caution_pit_df[["EventID", "DriverName", "PittedCautions", "PitStops", "PittedCautionPCT"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Strategy type
df = df.merge(
    strategy_type_df[["EventID", "DriverName", "TotalPitStops", "PitStrategy"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Position summary (best, worst, laps led, top5, top10, volatility)
df = df.merge(
    position_summary[["EventID", "DriverName", "BestRacePosition", "WorstRacePosition",
                       "LapsLed", "Top5Laps", "Top10Laps", "PositionVolatility"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Lap 1 positions
df = df.merge(
    lap1_positions[["EventID", "DriverName", "position_lap1"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained on pit cycles
df = df.merge(
    pit_cycle_gains[["EventID", "DriverName", "PositionsGainedPits"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained under caution
df = df.merge(
    caution_gains[["EventID", "DriverName", "PositionsGainedCaution"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace summary (fastest lap, consistency)
df = df.merge(
    pace_summary[["EventID", "DriverName", "BestLapSpeed", "LapConsistency"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace degradation
df = df.merge(
    pace_degradation[["EventID", "DriverName", "PaceDeg"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pace vs leader
df = df.merge(
    pace_vs_leader[["EventID", "DriverName", "AvgPaceVSLeadPace"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Pit duration
df = df.merge(
    pit_duration[["EventID", "DriverName", "AvgPitTime", "PitAvgTimeVSOverall"]],
    on=["EventID", "DriverName"],
    how="left"
)

# Positions gained lap 1 (needs PositionStart which is already in df)
df["PositionsGainedLap1"] = df["PositionStart"] - df["position_lap1"]

df["StartVsField"] = df["PositionStart"] / df["FieldSize"]

In [ ]:
df = df.sort_values(["EventDate", "EventID", "DriverID"]).reset_index(drop=True)

def compute_track_avg_pit_stops(df):
    result = []
    track_stop_history = {}

    for _, row in df.iterrows():
        track = row["TrackID"]

        if track in track_stop_history and len(track_stop_history[track]) > 0:
            result.append(np.mean(track_stop_history[track]))
        else:
            result.append(np.nan)

        if pd.notna(row["TotalPitStops"]):
            if track not in track_stop_history:
                track_stop_history[track] = []
            track_stop_history[track].append(row["TotalPitStops"])

    return result

df["TrackAvgPitStops"] = compute_track_avg_pit_stops(df)

df["FuelWindowEstimate"] = df["TotalRaceLaps"] / df["TrackAvgPitStops"]

def compute_track_avg_cautions(df):
    result = []
    track_caution_history = {}

    seen_events = set()

    for _, row in df.iterrows():
        track = row["TrackID"]
        event = row["EventID"]

        if track in track_caution_history and len(track_caution_history[track]) > 0:
            result.append(np.mean(track_caution_history[track]))
        else:
            result.append(np.nan)

        if event not in seen_events and pd.notna(row["TotalCautions"]):
            if track not in track_caution_history:
                track_caution_history[track] = []
            track_caution_history[track].append(row["TotalCautions"])
            seen_events.add(event)

    return result

df["TrackAvgCautions"] = compute_track_avg_cautions(df)

def compute_track_avg_caution_laps(df):
    result = []
    track_caution_lap_history = {}
    seen_events = set()

    for _, row in df.iterrows():
        track = row["TrackID"]
        event = row["EventID"]

        if track in track_caution_lap_history and len(track_caution_lap_history[track]) > 0:
            result.append(np.mean(track_caution_lap_history[track]))
        else:
            result.append(np.nan)

        if event not in seen_events and pd.notna(row["TotalCautionLaps"]):
            if track not in track_caution_lap_history:
                track_caution_lap_history[track] = []
            track_caution_lap_history[track].append(row["TotalCautionLaps"])
            seen_events.add(event)

    return result

df["TrackAvgCautionLaps"] = compute_track_avg_caution_laps(df)

def compute_track_type_avg_cautions(df):
    result = []
    type_caution_history = {}
    seen_events = set()

    for _, row in df.iterrows():
        track_type = row["EventTrackTypeID"]
        event = row["EventID"]

        if track_type in type_caution_history and len(type_caution_history[track_type]) > 0:
            result.append(np.mean(type_caution_history[track_type]))
        else:
            result.append(np.nan)

        if event not in seen_events and pd.notna(row["TotalCautions"]):
            if track_type not in type_caution_history:
                type_caution_history[track_type] = []
            type_caution_history[track_type].append(row["TotalCautions"])
            seen_events.add(event)

    return result

df["TrackTypeAvgCautions"] = compute_track_type_avg_cautions(df)

def compute_track_type_avg_caution_laps(df):
    result = []
    type_caution_lap_history = {}
    seen_events = set()

    for _, row in df.iterrows():
        track_type = row["EventTrackTypeID"]
        event = row["EventID"]

        if track_type in type_caution_lap_history and len(type_caution_lap_history[track_type]) > 0:
            result.append(np.mean(type_caution_lap_history[track_type]))
        else:
            result.append(np.nan)

        if event not in seen_events and pd.notna(row["TotalCautionLaps"]):
            if track_type not in type_caution_lap_history:
                type_caution_lap_history[track_type] = []
            type_caution_lap_history[track_type].append(row["TotalCautionLaps"])
            seen_events.add(event)

    return result

df["TrackTypeAvgCautionLaps"] = compute_track_type_avg_caution_laps(df)

def compute_track_avg_speed(df):
    result = []
    track_speed_history = {}
    seen_events = set()

    for _, row in df.iterrows():
        track = row["TrackID"]
        event = row["EventID"]

        if track in track_speed_history and len(track_speed_history[track]) > 0:
            result.append(np.mean(track_speed_history[track]))
        else:
            result.append(np.nan)

        if event not in seen_events and pd.notna(row["AvgRaceSpeed"]):
            if track not in track_speed_history:
                track_speed_history[track] = []
            track_speed_history[track].append(row["AvgRaceSpeed"])
            seen_events.add(event)

    return result

df["TrackAvgSpeed"] = compute_track_avg_speed(df)

def compute_team_track_avg_pit_stops(df):
    result = []
    history = {}

    for _, row in df.iterrows():
        key = (row["TeamID"], row["TrackID"])

        if key in history and len(history[key]) > 0:
            result.append(np.mean(history[key]))
        else:
            result.append(np.nan)

        if pd.notna(row["TotalPitStops"]):
            if key not in history:
                history[key] = []
            history[key].append(row["TotalPitStops"])

    return result

df["TeamTrackAvgPitStops"] = compute_team_track_avg_pit_stops(df)

def compute_team_track_avg_pitted_caution_pct(df):
    result = []
    history = {}

    for _, row in df.iterrows():
        key = (row["TeamID"], row["TrackID"])

        if key in history and len(history[key]) > 0:
            result.append(np.mean(history[key]))
        else:
            result.append(np.nan)

        if pd.notna(row["PittedCautionPCT"]):
            if key not in history:
                history[key] = []
            history[key].append(row["PittedCautionPCT"])

    return result

df["TeamTrackAvgPittedCautionPCT"] = compute_team_track_avg_pitted_caution_pct(df)

In [ ]:
df = df.sort_values(["EventDate", "EventID", "DriverID"]).reset_index(drop=True)
df = df[ordered_cols]

df.to_csv(BASE_DIR / "IndyCar_dataset_v25.csv")